In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Configuration
MAX_RUNTIME_HOURS = 11.5
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600
EPOCHS = 25
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
print(f"Maximum runtime: {MAX_RUNTIME_HOURS} hours")

# Create results directory
Path("results").mkdir(exist_ok=True)

# ==================== DATASET ====================
def get_data_loaders():
    try:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        
        trainset = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=transform
        )
        testset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=transform
        )
        
        trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        print(f"✓ Dataset loaded: {len(trainset)} train, {len(testset)} test samples")
        return trainloader, testloader
    except Exception as e:
        print(f"✗ Error loading dataset: {e}")
        raise

# ==================== VISION TRANSFORMER ====================
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class MLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, use_norm=True, use_res=True):
        super().__init__()
        self.use_norm = use_norm
        self.use_res = use_res
        
        self.norm1 = nn.LayerNorm(dim) if use_norm else nn.Identity()
        self.attn = MultiHeadAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim) if use_norm else nn.Identity()
        self.mlp = MLP(dim, int(dim * mlp_ratio))
        
    def forward(self, x):
        # Attention block
        if self.use_res:
            x = x + self.attn(self.norm1(x))
        else:
            x = self.attn(self.norm1(x))
        
        # MLP block
        if self.use_res:
            x = x + self.mlp(self.norm2(x))
        else:
            x = self.mlp(self.norm2(x))
        
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, num_classes=10, embed_dim=192, 
                 depth=12, num_heads=3, mlp_ratio=4.0, use_norm=True, use_res=True):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + 1, embed_dim))
        
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, use_norm, use_res)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim) if use_norm else nn.Identity()
        self.head = nn.Linear(embed_dim, num_classes)
        
        self._init_weights()
        
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
                    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        x = x[:, 0]  # CLS token
        x = self.head(x)
        return x

# ==================== METRICS LOGGER ====================
class MetricsLogger:
    def __init__(self, model, model_name):
        self.model = model
        self.model_name = model_name
        self.metrics = []
        self.hooks = []
        self.activations = {}
        
        # Register hooks
        for idx, block in enumerate(model.blocks):
            hook = block.mlp.register_forward_hook(self._make_hook(f'mlp_{idx}'))
            self.hooks.append(hook)
    
    def _make_hook(self, name):
        def hook(module, input, output):
            self.activations[name] = output.detach()
        return hook
    
    def compute_stable_rank(self, x):
        """Compute stable rank: ||X||_F^2 / ||X||_2^2"""
        try:
            if x.dim() == 3:
                B, N, D = x.shape
                x = x.reshape(B * N, D)
            
            # Handle all-zero or near-zero matrices
            if x.abs().max() < 1e-8:
                return 1.0
            
            # Compute SVD
            s = torch.linalg.svdvals(x.float())
            
            # Filter out very small singular values
            s = s[s > 1e-8]
            
            if len(s) == 0:
                return 1.0
            
            frobenius_norm_sq = (s ** 2).sum()
            spectral_norm_sq = s[0] ** 2
            
            if spectral_norm_sq < 1e-8:
                return 1.0
            
            stable_rank = (frobenius_norm_sq / spectral_norm_sq).item()
            return max(1.0, stable_rank)
        
        except Exception as e:
            print(f"Warning: Error computing stable rank: {e}")
            return 1.0
    
    def compute_active_neurons(self, x):
        """Compute percentage of active neurons (> 0)"""
        try:
            return (x > 0).float().mean().item()
        except:
            return 0.0
    
    def compute_gradient_norm(self):
        """Compute L2 norm of gradients for MLP layers"""
        try:
            grad_norms = []
            for idx, block in enumerate(self.model.blocks):
                if block.mlp.fc1.weight.grad is not None:
                    norm = block.mlp.fc1.weight.grad.norm().item()
                    grad_norms.append(norm)
            return np.mean(grad_norms) if grad_norms else 0.0
        except:
            return 0.0
    
    def log_metrics(self, epoch, train_loss, val_loss, val_acc):
        """Compute and log all metrics"""
        try:
            metrics_dict = {
                'model': self.model_name,
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_acc': val_acc,
                'grad_norm': self.compute_gradient_norm()
            }
            
            # Compute stable rank and active neurons for each layer
            for name, activation in self.activations.items():
                layer_idx = name.split('_')[1]
                rank = self.compute_stable_rank(activation)
                active = self.compute_active_neurons(activation)
                
                metrics_dict[f'rank_layer_{layer_idx}'] = rank
                metrics_dict[f'active_layer_{layer_idx}'] = active
            
            self.metrics.append(metrics_dict)
            self.activations.clear()  # Clear for next epoch
            
            return metrics_dict
        except Exception as e:
            print(f"Warning: Error logging metrics: {e}")
            return None
    
    def get_dataframe(self):
        return pd.DataFrame(self.metrics)
    
    def cleanup(self):
        for hook in self.hooks:
            hook.remove()

# ==================== TRAINING FUNCTIONS ====================
def train_epoch(model, trainloader, optimizer, criterion, device, metric_batches=5):
    model.train()
    total_loss = 0
    processed_batches = 0
    
    try:
        for batch_idx, (data, target) in enumerate(trainloader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            processed_batches += 1
            
            # Only process first few batches for metrics (speed optimization)
            if batch_idx >= metric_batches:
                break
        
        return total_loss / processed_batches if processed_batches > 0 else 0.0
    except Exception as e:
        print(f"Error in training: {e}")
        return 0.0

@torch.no_grad()
def validate(model, testloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    try:
        for data, target in testloader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
        
        avg_loss = total_loss / len(testloader)
        accuracy = 100. * correct / total
        return avg_loss, accuracy
    except Exception as e:
        print(f"Error in validation: {e}")
        return 0.0, 0.0

# ==================== PLOTTING FUNCTIONS ====================
def save_plots(all_metrics, start_time):
    """Generate and save all plots"""
    try:
        print("\n📊 Generating plots...")
        
        # Plot A: Heatmap (Waterfall)
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.flatten()
        
        for idx, (model_name, df) in enumerate(all_metrics.items()):
            ax = axes[idx]
            
            # Extract rank columns
            rank_cols = [col for col in df.columns if col.startswith('rank_layer_')]
            if len(rank_cols) == 0:
                continue
            
            rank_data = df[rank_cols].values.T
            
            sns.heatmap(rank_data, ax=ax, cmap='viridis', cbar_kws={'label': 'Stable Rank'},
                       xticklabels=df['epoch'].values, yticklabels=range(len(rank_cols)))
            ax.set_title(f'{model_name} - Rank Evolution', fontsize=14, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Layer Index')
        
        plt.tight_layout()
        plt.savefig('results/plot_A_heatmap.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✓ Plot A saved: plot_A_heatmap.png")
        
        # Plot B: Depth Slice (Line Plot)
        fig, ax = plt.subplots(figsize=(12, 6))
        
        for model_name, df in all_metrics.items():
            if len(df) == 0:
                continue
            last_epoch = df.iloc[-1]
            rank_cols = [col for col in df.columns if col.startswith('rank_layer_')]
            ranks = [last_epoch[col] for col in sorted(rank_cols, key=lambda x: int(x.split('_')[-1]))]
            layers = list(range(len(ranks)))
            ax.plot(layers, ranks, marker='o', label=model_name, linewidth=2)
        
        ax.set_xlabel('Layer Index', fontsize=12)
        ax.set_ylabel('Stable Rank', fontsize=12)
        ax.set_title('Stable Rank by Layer (Final Epoch)', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('results/plot_B_depth_slice.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✓ Plot B saved: plot_B_depth_slice.png")
        
        # Plot C: Rank vs Sparsity (ViT_Naked only)
        if 'ViT_Naked' in all_metrics:
            fig, ax1 = plt.subplots(figsize=(12, 6))
            df = all_metrics['ViT_Naked']
            
            if len(df) > 0:
                # Average rank across all layers
                rank_cols = [col for col in df.columns if col.startswith('rank_layer_')]
                df['avg_rank'] = df[rank_cols].mean(axis=1)
                
                # Average active neurons
                active_cols = [col for col in df.columns if col.startswith('active_layer_')]
                df['avg_active'] = df[active_cols].mean(axis=1)
                
                ax1.plot(df['epoch'], df['avg_rank'], 'b-', label='Avg Stable Rank', linewidth=2)
                ax1.set_xlabel('Epoch', fontsize=12)
                ax1.set_ylabel('Average Stable Rank', color='b', fontsize=12)
                ax1.tick_params(axis='y', labelcolor='b')
                
                ax2 = ax1.twinx()
                ax2.plot(df['epoch'], df['avg_active'] * 100, 'r-', label='Active Neurons %', linewidth=2)
                ax2.set_ylabel('Active Neurons (%)', color='r', fontsize=12)
                ax2.tick_params(axis='y', labelcolor='r')
                
                ax1.set_title('ViT_Naked: Rank vs Sparsity', fontsize=14, fontweight='bold')
                ax1.grid(True, alpha=0.3)
                
                fig.tight_layout()
                plt.savefig('results/plot_C_rank_vs_sparsity.png', dpi=150, bbox_inches='tight')
                plt.close()
                print("  ✓ Plot C saved: plot_C_rank_vs_sparsity.png")
        
        # Plot D: Accuracy Validation
        fig, ax = plt.subplots(figsize=(12, 6))
        
        for model_name, df in all_metrics.items():
            if len(df) > 0:
                ax.plot(df['epoch'], df['val_acc'], marker='o', label=model_name, linewidth=2)
        
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
        ax.set_title('Validation Accuracy Over Time', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.axhline(y=10, color='gray', linestyle='--', label='Chance Level (10%)')
        plt.tight_layout()
        plt.savefig('results/plot_D_accuracy.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("  ✓ Plot D saved: plot_D_accuracy.png")
        
        print("✓ All plots generated successfully!")
        
    except Exception as e:
        print(f"✗ Error generating plots: {e}")

# ==================== MAIN EXPERIMENT ====================
def run_experiment():
    global_start_time = time.time()
    
    print("="*80)
    print(" TRANSFORMER COLLAPSE PROTOCOL - EXPERIMENT 1")
    print("="*80)
    
    # Load data
    try:
        trainloader, testloader = get_data_loaders()
    except Exception as e:
        print(f"Failed to load data: {e}")
        return
    
    # Model configurations
    configs = {
        'ViT_Star': {'use_norm': True, 'use_res': True},
        'ViT_NoNorm': {'use_norm': False, 'use_res': True},
        'ViT_NoRes': {'use_norm': True, 'use_res': False},
        'ViT_Naked': {'use_norm': False, 'use_res': False},
    }
    
    all_metrics = {}
    
    for model_idx, (model_name, config) in enumerate(configs.items()):
        # Check time budget
        elapsed = time.time() - global_start_time
        if elapsed > MAX_RUNTIME_SECONDS:
            print(f"\n⏰ Time limit reached! Stopping experiment and saving results...")
            break
        
        remaining_time = (MAX_RUNTIME_SECONDS - elapsed) / 3600
        print(f"\n{'='*80}")
        print(f" MODEL {model_idx+1}/4: {model_name}")
        print(f" Remaining time: {remaining_time:.2f} hours")
        print(f"{'='*80}")
        
        try:
            # Initialize model
            model = VisionTransformer(
                img_size=32, patch_size=4, num_classes=10,
                embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0,
                use_norm=config['use_norm'], use_res=config['use_res']
            ).to(DEVICE)
            
            optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            criterion = nn.CrossEntropyLoss()
            logger = MetricsLogger(model, model_name)
            
            model_start_time = time.time()
            
            # Training loop
            for epoch in range(1, EPOCHS + 1):
                epoch_start = time.time()
                
                # Check time budget
                elapsed = time.time() - global_start_time
                if elapsed > MAX_RUNTIME_SECONDS:
                    print(f"\n⏰ Time limit reached at epoch {epoch}!")
                    break
                
                # Train
                train_loss = train_epoch(model, trainloader, optimizer, criterion, DEVICE)
                
                # Validate
                val_loss, val_acc = validate(model, testloader, criterion, DEVICE)
                
                # Log metrics
                metrics = logger.log_metrics(epoch, train_loss, val_loss, val_acc)
                
                epoch_time = time.time() - epoch_start
                print(f"Epoch {epoch:2d}/{EPOCHS} | "
                      f"Train Loss: {train_loss:.4f} | "
                      f"Val Loss: {val_loss:.4f} | "
                      f"Val Acc: {val_acc:.2f}% | "
                      f"Time: {epoch_time:.1f}s")
                
                # Save intermediate results
                if epoch % 5 == 0 or epoch == EPOCHS:
                    df = logger.get_dataframe()
                    df.to_csv(f'results/{model_name}_metrics.csv', index=False)
            
            # Final save for this model
            df = logger.get_dataframe()
            df.to_csv(f'results/{model_name}_metrics.csv', index=False)
            all_metrics[model_name] = df
            
            model_time = time.time() - model_start_time
            print(f"✓ {model_name} completed in {model_time/60:.2f} minutes")
            
            # Cleanup
            logger.cleanup()
            del model, optimizer, logger
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"✗ Error training {model_name}: {e}")
            continue
    
    # Generate plots with all available data
    try:
        save_plots(all_metrics, global_start_time)
    except Exception as e:
        print(f"Error generating final plots: {e}")
    
    # Save combined metrics
    try:
        combined_df = pd.concat([df for df in all_metrics.values()], ignore_index=True)
        combined_df.to_csv('results/all_metrics.csv', index=False)
        print(f"\n✓ Combined metrics saved: results/all_metrics.csv")
    except Exception as e:
        print(f"Warning: Could not save combined metrics: {e}")
    
    total_time = time.time() - global_start_time
    print(f"\n{'='*80}")
    print(f" EXPERIMENT COMPLETED")
    print(f" Total Runtime: {total_time/3600:.2f} hours ({total_time/60:.1f} minutes)")
    print(f" Results saved in: ./results/")
    print(f"{'='*80}")

# ==================== EXECUTION ====================
if __name__ == "__main__":
    run_experiment()

Using device: cuda
Maximum runtime: 11.5 hours
 TRANSFORMER COLLAPSE PROTOCOL - EXPERIMENT 1


100%|██████████| 170M/170M [00:03<00:00, 47.7MB/s]


✓ Dataset loaded: 50000 train, 10000 test samples

 MODEL 1/4: ViT_Star
 Remaining time: 11.50 hours
Epoch  1/25 | Train Loss: 3.6885 | Val Loss: 2.4767 | Val Acc: 14.34% | Time: 4.1s
Epoch  2/25 | Train Loss: 2.4126 | Val Loss: 2.3389 | Val Acc: 14.22% | Time: 2.8s
Epoch  3/25 | Train Loss: 2.2765 | Val Loss: 2.2277 | Val Acc: 17.84% | Time: 2.8s
Epoch  4/25 | Train Loss: 2.2541 | Val Loss: 2.1303 | Val Acc: 19.52% | Time: 2.8s
Epoch  5/25 | Train Loss: 2.1723 | Val Loss: 2.1364 | Val Acc: 20.16% | Time: 2.8s
Epoch  6/25 | Train Loss: 2.0960 | Val Loss: 2.0775 | Val Acc: 18.41% | Time: 2.8s
Epoch  7/25 | Train Loss: 2.0473 | Val Loss: 2.0309 | Val Acc: 22.54% | Time: 2.8s
Epoch  8/25 | Train Loss: 2.0198 | Val Loss: 2.0110 | Val Acc: 22.12% | Time: 2.8s
Epoch  9/25 | Train Loss: 2.0385 | Val Loss: 1.9904 | Val Acc: 24.99% | Time: 2.8s
Epoch 10/25 | Train Loss: 2.0375 | Val Loss: 2.0275 | Val Acc: 22.96% | Time: 2.8s
Epoch 11/25 | Train Loss: 2.0215 | Val Loss: 1.9499 | Val Acc: 25.82%